# Semana 9 · ¿Qué hace falta para cambiar un movimiento?

Este cuaderno acompaña a la sesión de proyecto de la semana 9: simular una caída libre y un lanzamiento con vectores, y comprobar cómo el tamaño del paso afecta al error.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar.

Tiempo estimado: 30-45 minutos.

## 1. Caída libre

Cerca de la superficie, un cuerpo que cae sin rozamiento lo hace con aceleración constante $g = 9{,}80665$ m/s² (valor estándar). La altura recorrida es una función cuadrática del tiempo:

$$y(t) = \tfrac{1}{2}\,g\,t^2, \qquad v(t) = g\,t.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

g = 9.80665       # m/s², aceleración estándar de la gravedad

def caida_exacta(t):
    return 0.5 * g * t**2, g * t      # distancia recorrida y velocidad

altura = 20.0                          # m, altura desde la que se suelta
t_caida = np.sqrt(2 * altura / g)
print(f"Tiempo de caída desde {altura} m: {t_caida:.2f} s   velocidad al llegar: {g * t_caida:.1f} m/s = {g * t_caida * 3.6:.0f} km/h")

**Ejercicio.** ¿Cuánto tarda en llegar al suelo una piedra soltada desde 5 m? ¿Y desde 80 m?

In [ ]:
# Tu código aquí

In [ ]:
# Solución
for h in [5, 80]:
    print(h, "m:", round(np.sqrt(2 * h / g), 2), "s")

## 2. Simulación paso a paso (método de Euler)

En lugar de usar la fórmula, avanzamos el reloj $\Delta t$ cada vez, como en la semana 8: la velocidad cambia con la aceleración y la posición con la velocidad. Es el **método de Euler**.

In [ ]:
def euler_caida(t_final, dt):
    t, y, v = 0.0, 0.0, 0.0
    while t < t_final - 1e-12:
        y += v * dt
        v += g * dt
        t += dt
    return y, v

T = 2.0
y_ex, v_ex = caida_exacta(T)
print(f"Exacto: y = {y_ex:.4f} m,  v = {v_ex:.4f} m/s")
for dt in [0.5, 0.1, 0.01, 0.001]:
    y_e, v_e = euler_caida(T, dt)
    print(f"dt = {dt:5.3f} s:  y = {y_e:8.4f} m   error en y = {y_e - y_ex:+.4f} m   error en v = {v_e - v_ex:+.5f} m/s")

Observa que el error en la posición es proporcional a $\Delta t$: dividir el paso entre 10 divide el error entre 10. La velocidad sale exacta porque la aceleración es constante.

## 3. Vectores y lanzamientos

Un lanzamiento con velocidad inicial $v_0$ y ángulo $\theta$ sobre la horizontal tiene componentes $v_{0x} = v_0\cos\theta$ y $v_{0y} = v_0\sin\theta$. En el eje horizontal no hay aceleración, y en el vertical actúa $-g$:

$$x(t) = v_{0x}\,t, \qquad y(t) = v_{0y}\,t - \tfrac{1}{2}\,g\,t^2.$$

In [ ]:
def trayectoria(v0, angulo, n=300):
    th = np.radians(angulo)
    vx, vy = v0 * np.cos(th), v0 * np.sin(th)
    t_vuelo = 2 * vy / g
    t = np.linspace(0, t_vuelo, n)
    return vx * t, vy * t - 0.5 * g * t**2, t_vuelo

fig, ax = plt.subplots(figsize=(9, 4))
for ang in [30, 45, 60]:
    x, y, tv = trayectoria(20, ang)
    ax.plot(x, y, label=f"{ang}°  (alcance {x[-1]:.1f} m, vuelo {tv:.2f} s)")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Lanzamientos a 20 m/s sin rozamiento")
plt.show()

De los tres, el de 45° es el que llega más lejos: el alcance $R = v_0^2 \sin(2\theta)/g$ es máximo cuando $\sin 2\theta = 1$, es decir, para 45°. Lo comprobamos.

## 4. Una comprobación que hizo Newton

La Luna no sale despedida: **cae** continuamente hacia la Tierra sin llegar a tocarla, porque su velocidad lateral hace que la trayectoria se curve a la vez. La aceleración que necesita para describir su órbita casi circular es $a = v^2/r$. Si la fuerza que la sostiene es la misma que hace caer una piedra, pero debilitada con el cuadrado de la distancia, esa aceleración debe ser $g/(r/R_\oplus)^2$.

In [ ]:
v_luna = 1.022e3         # m/s, velocidad orbital media de la Luna
r_luna = 3.844e8         # m, distancia media
R_tierra = 6.371e6       # m

a_orbita = v_luna**2 / r_luna
a_prevista = g / (r_luna / R_tierra) ** 2
print(f"Aceleración de la órbita lunar (v²/r):      {a_orbita:.5f} m/s²")
print(f"Aceleración prevista por g / (r/R)²:         {a_prevista:.5f} m/s²")
print(f"Diferencia: {100 * (a_orbita / a_prevista - 1):+.1f} %")

Las dos cifras coinciden con un error del 1 %. La misma fuerza que hace caer la manzana mantiene a la Luna en su órbita. En la fase 2 desarrollarás esta idea.

## 5. Para pensar

1. Cambia el paso del método de Euler a 0,0001 s. ¿Cuánto tarda en ejecutarse y cuánto mejora el error?
2. Calcula el alcance de un lanzamiento a 20 m/s y 45° con la fórmula $R = v_0^2\sin(2\theta)/g$ y compáralo con la simulación.
3. ¿Qué pasaría con las trayectorias de la gráfica si no hubiese gravedad?